# Regression Metrics

**Interview Focus**: MSE vs MAE tradeoffs, R-squared interpretation, residual analysis.

**Key Concepts**:
- MSE, RMSE, MAE, MAPE, R-squared, Adjusted R-squared
- Sensitivity to outliers
- Residual analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 1. Metrics from Scratch

In [ ]:
def mse(y_true, y_pred):
    """Mean Squared Error: average of squared residuals."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    """Root Mean Squared Error: sqrt(MSE). Same units as y."""
    return np.sqrt(mse(y_true, y_pred))

def mae(y_true, y_pred):
    """Mean Absolute Error: average of absolute residuals."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred))

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error.
    Warning: undefined when y_true has zeros."""
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    # Avoid division by zero
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def r_squared(y_true, y_pred):
    """R-squared (coefficient of determination).
    R^2 = 1 - SS_res / SS_tot
    SS_res = sum((y - y_hat)^2)    (residual sum of squares)
    SS_tot = sum((y - y_mean)^2)   (total sum of squares)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def adjusted_r_squared(y_true, y_pred, n_features):
    """Adjusted R-squared: penalizes for number of features.
    R^2_adj = 1 - (1 - R^2) * (n - 1) / (n - p - 1)
    where n = samples, p = features.
    """
    r2 = r_squared(y_true, y_pred)
    n = len(y_true)
    return 1 - (1 - r2) * (n - 1) / (n - n_features - 1)


# Generate sample data
np.random.seed(42)
n = 100
x = np.linspace(0, 10, n)
y_true_reg = 2 * x + 3 + np.random.normal(0, 2, n)
y_pred_reg = 2 * x + 3.5  # slightly off predictions

print("Regression Metrics:")
print(f"  MSE:       {mse(y_true_reg, y_pred_reg):.4f}")
print(f"  RMSE:      {rmse(y_true_reg, y_pred_reg):.4f}")
print(f"  MAE:       {mae(y_true_reg, y_pred_reg):.4f}")
print(f"  MAPE:      {mape(y_true_reg, y_pred_reg):.2f}%")
print(f"  R-squared: {r_squared(y_true_reg, y_pred_reg):.4f}")
print(f"  Adj R^2:   {adjusted_r_squared(y_true_reg, y_pred_reg, n_features=1):.4f}")

## 2. Sensitivity to Outliers

In [ ]:
# Demonstrate outlier sensitivity
np.random.seed(42)
y_true_clean = np.random.normal(10, 2, 100)
y_pred_clean = y_true_clean + np.random.normal(0, 0.5, 100)

# Add 3 outliers
y_true_outlier = y_true_clean.copy()
y_pred_outlier = y_pred_clean.copy()
y_pred_outlier[:3] += 50  # 3 predictions are way off

print(f"{'Metric':<10} {'Without Outliers':>18} {'With 3 Outliers':>18} {'Change':>10}")
print("-" * 60)

for name, fn in [('MSE', mse), ('RMSE', rmse), ('MAE', mae)]:
    v_clean = fn(y_true_clean, y_pred_clean)
    v_outlier = fn(y_true_outlier, y_pred_outlier)
    change = (v_outlier - v_clean) / v_clean * 100
    print(f"{name:<10} {v_clean:>18.4f} {v_outlier:>18.4f} {change:>9.0f}%")

print(f"\nMSE/RMSE are much more sensitive to outliers because they square the error.")
print(f"MAE is more robust (linear penalty).")

In [ ]:
# Visualize loss functions
errors = np.linspace(-10, 10, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss functions
axes[0].plot(errors, errors**2, 'b-', linewidth=2, label='MSE (squared)')
axes[0].plot(errors, np.abs(errors), 'r-', linewidth=2, label='MAE (absolute)')
axes[0].plot(errors, np.where(np.abs(errors) < 1, 0.5*errors**2, np.abs(errors)-0.5),
         'g--', linewidth=2, label='Huber (delta=1)')
axes[0].set_xlabel('Prediction Error')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Functions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 30)

# Gradients
axes[1].plot(errors, 2*errors, 'b-', linewidth=2, label='MSE gradient (2*error)')
axes[1].plot(errors, np.sign(errors), 'r-', linewidth=2, label='MAE gradient (sign)')
axes[1].set_xlabel('Prediction Error')
axes[1].set_ylabel('Gradient')
axes[1].set_title('Gradients')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("MSE gradient grows linearly with error → large errors dominate training.")
print("MAE gradient is constant → outliers don't dominate, but not differentiable at 0.")
print("Huber loss: MSE for small errors, MAE for large errors (best of both).")

## 3. R-squared Can Be Negative

In [ ]:
# R^2 is negative when the model is WORSE than predicting the mean
np.random.seed(42)
y_true_r2 = np.array([3, 5, 7, 9, 11])

# Good prediction
y_pred_good = np.array([3.2, 4.8, 7.1, 9.3, 10.8])

# Bad prediction (worse than mean)
y_pred_bad = np.array([10, 1, 12, 2, 15])

# Mean prediction (baseline)
y_pred_mean = np.full_like(y_true_r2, np.mean(y_true_r2), dtype=float)

print(f"y_true:     {y_true_r2}")
print(f"y_mean:     {y_pred_mean}")
print(f"y_good:     {y_pred_good}")
print(f"y_bad:      {y_pred_bad}")
print()
print(f"R^2 (good):  {r_squared(y_true_r2, y_pred_good):.4f}  (close to 1 = excellent)")
print(f"R^2 (mean):  {r_squared(y_true_r2, y_pred_mean):.4f}  (always 0 for mean prediction)")
print(f"R^2 (bad):   {r_squared(y_true_r2, y_pred_bad):.4f}  (negative = worse than mean!)")
print()
print("R^2 interpretation:")
print("  1.0:  Perfect predictions")
print("  0.0:  Model = predicting the mean (baseline)")
print("  < 0:  Model is worse than the mean (something is wrong)")

## 4. Adjusted R-squared

In [ ]:
# Demonstrate why adjusted R^2 is needed
# Adding random features always increases R^2, even if they're noise

np.random.seed(42)
n_samples = 50
X_real = np.random.randn(n_samples, 1)
y_adj = 3 * X_real.squeeze() + np.random.randn(n_samples) * 0.5

r2_values = []
adj_r2_values = []
n_features_list = range(1, 40)

for n_feat in n_features_list:
    # Add random noise features
    X_noise = np.random.randn(n_samples, n_feat)
    X_full = np.hstack([X_real, X_noise]) if n_feat > 1 else X_real
    
    # Fit OLS
    X_with_bias = np.hstack([np.ones((n_samples, 1)), X_full])
    # Normal equation: w = (X^T X)^{-1} X^T y
    try:
        w = np.linalg.solve(X_with_bias.T @ X_with_bias, X_with_bias.T @ y_adj)
        y_pred_adj = X_with_bias @ w
        r2 = r_squared(y_adj, y_pred_adj)
        adj_r2 = adjusted_r_squared(y_adj, y_pred_adj, n_feat)
    except np.linalg.LinAlgError:
        r2 = r2_values[-1] if r2_values else 0
        adj_r2 = adj_r2_values[-1] if adj_r2_values else 0
    
    r2_values.append(r2)
    adj_r2_values.append(adj_r2)

plt.figure(figsize=(10, 5))
plt.plot(list(n_features_list), r2_values, 'b-o', linewidth=2, markersize=3, label='R-squared')
plt.plot(list(n_features_list), adj_r2_values, 'r-s', linewidth=2, markersize=3, label='Adjusted R-squared')
plt.axhline(y=r2_values[0], color='gray', linestyle='--', alpha=0.5, label='True model R^2')
plt.xlabel('Number of Features (1 real + noise)')
plt.ylabel('Score')
plt.title('R-squared vs Adjusted R-squared (Adding Noise Features)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("R^2 always increases with more features (even random noise).")
print("Adjusted R^2 penalizes for unnecessary features and can decrease.")

## 5. Residual Analysis

In [ ]:
# Generate data with different patterns
np.random.seed(42)
n = 200
x_ra = np.linspace(0, 10, n)

# Good model: random residuals
y_true_good = 2*x_ra + 3 + np.random.normal(0, 1.5, n)
y_pred_good_ra = 2*x_ra + 3
residuals_good = y_true_good - y_pred_good_ra

# Bad model: missing quadratic term (pattern in residuals)
y_true_quad = 0.3*x_ra**2 + 2*x_ra + 3 + np.random.normal(0, 1.5, n)
y_pred_linear = 5*x_ra + 1  # linear fit to quadratic data
residuals_bad = y_true_quad - y_pred_linear

# Heteroscedastic residuals (variance increases with x)
y_true_hetero = 2*x_ra + 3 + np.random.normal(0, 0.3*x_ra, n)
y_pred_hetero = 2*x_ra + 3
residuals_hetero = y_true_hetero - y_pred_hetero

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

datasets = [
    (y_pred_good_ra, residuals_good, 'Good Model\n(Random Residuals)'),
    (y_pred_linear, residuals_bad, 'Bad Model\n(Missing Nonlinearity)'),
    (y_pred_hetero, residuals_hetero, 'Heteroscedastic\n(Non-constant Variance)'),
]

for idx, (y_pred_plot, resid, title) in enumerate(datasets):
    # Residual plot
    axes[0, idx].scatter(y_pred_plot, resid, alpha=0.5, s=10)
    axes[0, idx].axhline(y=0, color='red', linestyle='--')
    axes[0, idx].set_xlabel('Predicted')
    axes[0, idx].set_ylabel('Residual')
    axes[0, idx].set_title(title)
    axes[0, idx].grid(True, alpha=0.3)
    
    # Q-Q plot
    stats.probplot(resid, dist="norm", plot=axes[1, idx])
    axes[1, idx].set_title(f'Q-Q Plot')
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("What to look for in residual plots:")
print("  1. Random scatter around 0 → good model")
print("  2. Pattern (curve, funnel) → model is missing something")
print("  3. Increasing spread → heteroscedasticity (violates OLS assumptions)")
print("  4. Q-Q plot: points on the line → normally distributed residuals")

## 6. Metric Comparison Summary

In [ ]:
# Side-by-side metric behavior
np.random.seed(42)
y_true_comp = np.random.uniform(5, 15, 100)

# Different levels of noise
noise_levels = np.linspace(0.1, 5, 20)

metrics_by_noise = {'MSE': [], 'RMSE': [], 'MAE': [], 'R^2': []}

for noise in noise_levels:
    y_pred_comp = y_true_comp + np.random.normal(0, noise, 100)
    metrics_by_noise['MSE'].append(mse(y_true_comp, y_pred_comp))
    metrics_by_noise['RMSE'].append(rmse(y_true_comp, y_pred_comp))
    metrics_by_noise['MAE'].append(mae(y_true_comp, y_pred_comp))
    metrics_by_noise['R^2'].append(r_squared(y_true_comp, y_pred_comp))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(noise_levels, metrics_by_noise['MSE'], 'b-', linewidth=2, label='MSE')
axes[0].plot(noise_levels, metrics_by_noise['RMSE'], 'r-', linewidth=2, label='RMSE')
axes[0].plot(noise_levels, metrics_by_noise['MAE'], 'g-', linewidth=2, label='MAE')
axes[0].set_xlabel('Noise Level (std)')
axes[0].set_ylabel('Metric Value')
axes[0].set_title('Error Metrics vs Noise Level')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(noise_levels, metrics_by_noise['R^2'], 'purple', linewidth=2)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Noise Level (std)')
axes[1].set_ylabel('R-squared')
axes[1].set_title('R-squared vs Noise Level')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interview Questions & Answers

---

**Q: When MAE over MSE?**

- **Use MAE** when outliers should not dominate the metric (e.g., housing prices with a few mansions). MAE penalizes all errors linearly.
- **Use MSE** when large errors are disproportionately bad (e.g., navigation, safety-critical). MSE penalizes large errors quadratically.
- **Use RMSE** for interpretability (same units as target) while keeping MSE's outlier sensitivity.
- **Use Huber loss** for training: MSE near zero, MAE far from zero. Best of both worlds.

---

**Q: R-squared can be negative — when?**

R^2 = 1 - SS_res/SS_tot. It's negative when SS_res > SS_tot, meaning the model's predictions are worse than simply predicting the mean of y. This happens when:
- The model is completely wrong
- Evaluating on a test set from a different distribution
- The model is badly overfit and fails on new data

R^2 = 0 means the model equals the mean baseline. R^2 = 1 means perfect prediction.

---

**Q: Adjusted R-squared — why?**

R^2 always increases when you add features (even random noise), because more parameters can always fit the training data better. Adjusted R^2 penalizes for the number of features:

```
R^2_adj = 1 - (1 - R^2) * (n-1) / (n-p-1)
```

It decreases if a new feature doesn't improve the model enough to justify the added complexity. Use it for feature selection.

---

**Q: MAPE limitations?**

- Undefined when y_true = 0
- Asymmetric: penalizes overestimates more than underestimates
- Use SMAPE (symmetric MAPE) or log-based metrics as alternatives